# How far ahead is the best player?

A ranking tells you the order. It does not tell you the *gap*. Second place
might be a whisker behind or a chasm.

This chapter measures the gap properly, by looking at where every player sits in
the distribution and asking how many standard deviations separate the best from
everyone else.

In [ ]:
import warnings

import matplotlib
import pandas as pd

from gambeta import needs, tifo

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")
keepers = pd.read_parquet(f"{SAMPLE}/keeper_ranking.parquet")

scores = ranking["score"].to_numpy(dtype=float)
mu, sigma = scores.mean(), scores.std(ddof=0)
print(f"players: {len(scores):,}")
print(f"mean:    {mu:.3f}")
print(f"sd:      {sigma:.3f}")

## The distribution

Most players cluster near the middle. That is what a composite of eleven
standardised requirements should do — by construction the average player scores
about zero.

The interesting part is the right tail.

In [ ]:
top5 = ranking.head(5)
fig = tifo.bell(
    ranking["score"],
    highlight=dict(zip(top5["player"], top5["score"], strict=True)),
    title="Every player, and how far out the best sit",
)

## Measuring the gap in sigma

A standard deviation is a natural yardstick here: it says how unusual a score is
*relative to the spread of the population itself*, so it travels across
different metrics and different eras.

In [ ]:
head = ranking.head(10).copy()
head["sigma"] = (head["score"] - mu) / sigma
head[["player", "score", "sigma", "seasons"]].round(2)

## The claim that should make you suspicious

Under a normal distribution, extreme values are astronomically rare. Let us take
that assumption seriously and see what it predicts.

In [ ]:
from math import erfc, sqrt

best = ranking.iloc[0]
z = (best["score"] - mu) / sigma
p = erfc(z / sqrt(2)) / 2

print(f"{best['player']} sits {z:.2f} standard deviations above the mean.")
print(f"Under a normal distribution, P(score >= that) = {p:.3e}")
print(f"That is roughly 1 player in {1 / p:,.0f}.")
print(f"\nOur population is {len(scores):,} players.")

**One in six hundred million, from a pool of five and a half thousand.**

Taken at face value this says the best player should not exist. Something is
wrong with the assumption, not with the footballer.

## The assumption is wrong: the distribution is not normal

A normal distribution is symmetric. Football talent is not.

In [ ]:
print(f"skewness: {pd.Series(scores).skew():.2f}   (0 = symmetric)")
print(f"kurtosis: {pd.Series(scores).kurtosis():.2f}   (0 = normal tails)")

for k in (2, 3, 4, 5, 6):
    expected = len(scores) * erfc(k / sqrt(2)) / 2
    actual = int((scores > mu + k * sigma).sum())
    print(f"  beyond {k} sigma: normal predicts {expected:8.2f}, we observe {actual:4d}")

The right tail is **much fatter than normal**. At three, four and five sigma
there are many more players than a bell curve allows.

This is not a defect in the data. It is a real property of elite performance,
and it has a name — a heavy-tailed distribution. Ability that compounds
(better players get better coaching, better teammates, more minutes, better
opponents to learn from) does not produce a symmetric bell. It produces a long
right tail where a handful of people are far beyond everyone else.

**So the honest statement is not "Messi is a 5.9-sigma player" as if that were a
probability.** It is: *under the wrong model he is impossible, and the fact that
he exists is evidence the model is wrong.*

## The gap between first and second

Sigma is still useful for comparison, as long as it is used as a ruler rather
than a probability.

In [ ]:
gaps = ranking.head(6)[["player", "score"]].copy()
gaps["sigma"] = (gaps["score"] - mu) / sigma
gaps["gap_to_next"] = gaps["score"].diff(-1)
gaps.round(2)

In [ ]:
first, second = ranking.iloc[0], ranking.iloc[1]
gap_sigma = (first["score"] - second["score"]) / sigma
print(f"{first['player']} to {second['player']}: {gap_sigma:.2f} sigma")
print(
    f"{second['player']} to {ranking.iloc[5]['player']}: "
    f"{(second['score'] - ranking.iloc[5]['score']) / sigma:.2f} sigma "
    f"(covering four players)"
)

## The result, stated in players rather than decimals

A lead of 0.39 means nothing on its own. The way to feel it is to take the gap
between first and second, lay it below someone else, and see **who you land
on**.

In [ ]:
q = ranking[ranking["qualified"]].reset_index(drop=True)
lead = q.loc[0, "score"] - q.loc[1, "score"]


def ordinal(n: int) -> str:
    suffix = "th" if 10 <= n % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


def named(i: int) -> str:
    return f"{q.loc[i, 'player']} ({ordinal(i + 1)})"


def same_distance_below(rank: int) -> int:
    """Index of the player sitting as far below `rank` as second sits below first."""
    target = q.loc[rank, "score"] - lead
    return int((q["score"] - target).abs().idxmin())


below_second = same_distance_below(1)
below_third = same_distance_below(2)

print(
    f"{named(1)} is as close to {named(0)}\n"
    f"  as {named(below_second)} is to {named(1)},\n"
    f"  or {named(below_third)} is to {named(2)}."
)
print(f"\n(every one of those gaps is {lead:.2f}, or {lead / sigma:.1f} sigma)")

Read that again, because it is the finding of this chapter. The distance from
second place to first is not a photo finish — it is the same distance that
separates second place from a player **six** positions further down, and third
place from one **six** positions down.

One more way to put it: compare the lead at the top against the entire spread of
the chasing pack.

In [ ]:
chasers = q.loc[1, "score"] - q.loc[7, "score"]
print(f"first to second:  {lead:.2f}")
print(f"second to eighth: {chasers:.2f}   (covering six players)")
print()
print(
    "The lead of one player over the next is larger than the spread"
    if lead > chasers
    else "The chasing pack is more spread out than the lead"
)
print("across the whole of the chasing pack.")

The top of this list is not a tight race followed by a drop-off. It is one
player clear, then a cluster.

## Where each requirement puts the best player

The composite hides which requirements the gap comes from. Breaking it out shows
whether dominance is broad or narrow.

In [ ]:
keys = [r.key for r in needs.OUTFIELD if r.key in ranking.columns]
profile = (
    ranking.set_index("player").loc[[ranking.iloc[0]["player"], ranking.iloc[1]["player"]], keys].T
)
profile.columns = [f"{c} (sigma)" for c in profile.columns]
profile.round(2)

## What would change my mind

- **A different composite.** These sigma figures are for the equal-weighted
  composite. Reweight the requirements and the gap changes.
- **The population defines the yardstick.** Sigma is measured against 5,508
  players who cleared the minutes threshold in the Big 5. Lower that threshold to
  include fringe players, or add a competition, and the standard deviation
  moves — which moves everyone's sigma.
- **Heavy tails cut both ways.** If the distribution is not normal, sigma is a
  descriptive ruler and nothing more. Any sentence of the form "this is a
  one-in-N-billion player" is misusing it, including one I could easily have
  written above.